In [2]:
!py -m pip install ultralytics datasets Pillow pyyaml


[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from datasets import load_dataset
from ultralytics import YOLO
from pathlib import Path
import yaml

c:\Users\wikto\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# ── Step 1: Inspect one sample to confirm structure ─────────

ds = load_dataset("Francesco/vehicles-q0x2v")
print(ds)
sample = ds["train"][0]
print("Keys:", sample.keys())
print("Objects:", sample["objects"])

DatasetDict({
    train: Dataset({
        features: ['image_id', 'image', 'width', 'height', 'objects'],
        num_rows: 2634
    })
    validation: Dataset({
        features: ['image_id', 'image', 'width', 'height', 'objects'],
        num_rows: 458
    })
    test: Dataset({
        features: ['image_id', 'image', 'width', 'height', 'objects'],
        num_rows: 966
    })
})
Keys: dict_keys(['image_id', 'image', 'width', 'height', 'objects'])
Objects: {'id': [30613, 30614, 30615, 30616, 30617, 30618], 'area': [960, 936, 5428, 5996, 2033, 3240], 'bbox': [[328.0, 203.0, 24.0, 40.0], [302.0, 243.0, 26.0, 36.0], [148.0, 226.0, 59.0, 92.0], [51.0, 259.0, 67.0, 89.5], [318.0, 285.0, 38.0, 53.5], [175.0, 326.0, 54.0, 60.0]], 'category': [11, 5, 1, 9, 5, 5]}


In [5]:
# ── Step 2: Convert to YOLO folder structure ─────────────────
ds = load_dataset("Francesco/vehicles-q0x2v")

ROOT = Path("vehicles_yolo")

def convert_split(split_name, hf_split):
    img_dir = ROOT / "images" / split_name
    lbl_dir = ROOT / "labels" / split_name
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)

    for i, sample in enumerate(hf_split):
        img   = sample["image"]          # PIL image
        W, H  = img.size                 # always 640x640 for this dataset

        # Save image as JPEG
        img.save(img_dir / f"{i:06d}.jpg")

        # Write YOLO label — one line per object: class cx cy w h (normalized)
        objects = sample["objects"]
        bboxes  = objects["bbox"]        # list of [x, y, w, h] in pixels (COCO)

        # Check if category labels exist (single-class → default to 0)
        labels = objects.get("category", [0] * len(bboxes))

        with open(lbl_dir / f"{i:06d}.txt", "w") as f:
            for bbox, cls in zip(bboxes, labels):
                x, y, w, h = bbox
                cx = (x + w / 2) / W
                cy = (y + h / 2) / H
                nw = w / W
                nh = h / H
                # Clamp to [0, 1] just in case of any float rounding
                cx, cy, nw, nh = (max(0.0, min(1.0, v)) for v in (cx, cy, nw, nh))
                f.write(f"{int(cls)} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}\n")

        if i % 200 == 0:
            print(f"  [{split_name}] {i}/{len(hf_split)}")

for hf_split, yolo_split in [("train","train"), ("validation","val"), ("test","test")]:
    print(f"Converting {hf_split}...")
    convert_split(yolo_split, ds[hf_split])

print("✅ Done!")

Converting train...
  [train] 0/2634
  [train] 200/2634
  [train] 400/2634
  [train] 600/2634
  [train] 800/2634
  [train] 1000/2634
  [train] 1200/2634
  [train] 1400/2634
  [train] 1600/2634
  [train] 1800/2634
  [train] 2000/2634
  [train] 2200/2634
  [train] 2400/2634
  [train] 2600/2634
Converting validation...
  [val] 0/458
  [val] 200/458
  [val] 400/458
Converting test...
  [test] 0/966
  [test] 200/966
  [test] 400/966
  [test] 600/966
  [test] 800/966
✅ Done!


In [6]:
# ── Step 3: Write dataset.yaml ───────────────────────────────

cfg = {
    "path":  str(ROOT.resolve()),
    "train": "images/train",
    "val":   "images/val",
    "test":  "images/test",
    "nc":    1,
    "names": ["vehicle"],   # update if you find multiple classes in Step 1
}

yaml_path = ROOT / "dataset.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(cfg, f, default_flow_style=False)

print(f"Saved: {yaml_path}")


Saved: vehicles_yolo\dataset.yaml


In [7]:
# ── Step 4a: Fine-tune YOLO26s ───────────────────────────────
model = YOLO("yolo26s.pt")

model.train(
    data=str(yaml_path),
    epochs=3,
    imgsz=320,
    batch=4,
    device="cpu",
    project="C:/yolo_runs",
    name="vehicles",
    save_period=-1,    # only save best + last, no intermediate checkpoints
    exist_ok=True,
)

Ultralytics 8.4.37  Python-3.11.0 torch-2.11.0+cpu CPU (11th Gen Intel Core i7-1165G7 @ 2.80GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=vehicles_yolo\dataset.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=3, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=vehicles, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, pat

c:\Users\wikto\AppData\Local\Programs\Python\Python311\Lib\site-packages\ultralytics\utils\metrics.py:839: RuntimeWarning: Mean of empty slice.
  i = smooth(f1_curve.mean(0), 0.1).argmax()  # max F1 index
c:\Users\wikto\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\core\_methods.py:121: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
        2/3         0G          0      1.741          0          0        320: 100% ━━━━━━━━━━━━ 1/1 2.5it/s 0.4s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 3.4it/s 0.3s
                   all          4          0          0          0          0          0
WARNING no labels found in detect set, cannot compute metrics without labels


c:\Users\wikto\AppData\Local\Programs\Python\Python311\Lib\site-packages\ultralytics\utils\metrics.py:839: RuntimeWarning: Mean of empty slice.
  i = smooth(f1_curve.mean(0), 0.1).argmax()  # max F1 index
c:\Users\wikto\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\core\_methods.py:121: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
        3/3         0G          0      1.737          0          0        320: 100% ━━━━━━━━━━━━ 1/1 2.6it/s 0.4s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 3.3it/s 0.3s
                   all          4          0          0          0          0          0
WARNING no labels found in detect set, cannot compute metrics without labels


c:\Users\wikto\AppData\Local\Programs\Python\Python311\Lib\site-packages\ultralytics\utils\metrics.py:839: RuntimeWarning: Mean of empty slice.
  i = smooth(f1_curve.mean(0), 0.1).argmax()  # max F1 index
c:\Users\wikto\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\core\_methods.py:121: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(



3 epochs completed in 0.001 hours.
Optimizer stripped from C:\yolo_runs\vehicles\weights\last.pt, 20.3MB
Optimizer stripped from C:\yolo_runs\vehicles\weights\best.pt, 20.3MB

Validating C:\yolo_runs\vehicles\weights\best.pt...
Ultralytics 8.4.37  Python-3.11.0 torch-2.11.0+cpu CPU (11th Gen Intel Core i7-1165G7 @ 2.80GHz)
YOLO26s summary (fused): 122 layers, 9,465,567 parameters, 0 gradients, 20.5 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 3.8it/s 0.3s


c:\Users\wikto\AppData\Local\Programs\Python\Python311\Lib\site-packages\ultralytics\utils\metrics.py:657: RuntimeWarning: Mean of empty slice.
  ax.plot(px, py.mean(1), linewidth=3, color="blue", label=f"all classes {ap[:, 0].mean():.3f} mAP@0.5")
c:\Users\wikto\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\core\_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\Users\wikto\AppData\Local\Programs\Python\Python311\Lib\site-packages\ultralytics\utils\metrics.py:703: RuntimeWarning: Mean of empty slice.
  y = smooth(py.mean(0), 0.1)
c:\Users\wikto\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\core\_methods.py:121: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
c:\Users\wikto\AppData\Local\Programs\Python\Python311\Lib\site-packages\ultralytics\utils\metrics.py:703: RuntimeWarning: Mean of empty slice.
  y = smooth(py.mean(0), 0.1)
c:\Users\wikto\AppData\Local\

                   all          4          0          0          0          0          0
WARNING no labels found in detect set, cannot compute metrics without labels
Speed: 0.3ms preprocess, 59.5ms inference, 0.0ms loss, 0.1ms postprocess per image
Results saved to C:\yolo_runs\vehicles


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([], dtype=int32)
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000022176026C90>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
    

In [8]:
best = YOLO("runs/vehicles/yolo26s/weights/best.pt")

metrics = best.val(data=str(yaml_path))
print(f"mAP50:     {metrics.box.map50:.3f}")
print(f"mAP50-95:  {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall:    {metrics.box.mr:.3f}")


FileNotFoundError: [Errno 2] No such file or directory: 'runs\\vehicles\\yolo26s\\weights\\best.pt'